In [ ]:
import tensorflow as tf

In [ ]:
import pandas as pd
heart_df=pd.read_csv('./datasets/heart.csv')
heart_df.head()

In [ ]:
pd.Series.max(heart_df['Oldpeak'])

In [ ]:
heart_df['Oldpeak'].value_counts()

In [ ]:
X=heart_df.iloc[:,:11]
y=heart_df.iloc[:,-1]
y

In [ ]:
X.head()

In [ ]:
cat=[]
num=[]
for col in X.columns:
    if len(heart_df[col].unique()) < 5:
        cat.append(col)
    else:
        num.append(col)

In [ ]:
cat, num

In [ ]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
for col in cat:
    X[col]=le.fit_transform(X[col])

In [ ]:
X

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3)

In [ ]:
X_train

In [ ]:
y.value_counts()

In [ ]:
from sklearn.preprocessing import MinMaxScaler
sc=MinMaxScaler()
for col in num:
    X_train[col]=sc.fit_transform(X_train[col].values.reshape(-1, 1))
    X_test[col]=sc.transform(X_test[col].values.reshape(-1, 1))

In [ ]:
X_test

In [ ]:
X_train.shape

In [ ]:
X_train

In [ ]:
tf.random.set_seed(42)

model=tf.keras.Sequential([
    tf.keras.layers.Dense(11,activation="relu"),
    tf.keras.layers.Dense(10,activation="relu"),
    tf.keras.layers.Dense(5,activation="relu"),
    tf.keras.layers.Dense(1,activation="sigmoid")
])

model.compile(loss=tf.keras.losses.BinaryCrossentropy(),
              optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
            # optimizer=tf.keras.optimizers.SGD(),
              metrics=["accuracy"])

history=model.fit(X_train,y_train,epochs=500,validation_data=(X_test,y_test))

In [ ]:
history=pd.DataFrame(history.history)
history.plot()

In [ ]:
y_pred=model.predict(X_test)

In [ ]:
y_pred

In [ ]:
# Note: The following confusion matrix code is a remix of Scikit-Learn's 
# plot_confusion_matrix function - https://scikit-learn.org/stable/modules/generated/sklearn.metrics.plot_confusion_matrix.html
# and Made with ML's introductory notebook - https://github.com/GokuMohandas/MadeWithML/blob/main/notebooks/08_Neural_Networks.ipynb 
import itertools
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline
from sklearn.metrics import confusion_matrix
def plot_confusion_matrix(y_test,y_pred,figsize=(5,5)):
  # Create the confusion matrix
  cm = confusion_matrix(y_test, tf.round(y_pred))
  cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis] # normalize it
  n_classes = cm.shape[0]

  # Let's prettify it
  fig, ax = plt.subplots(figsize=figsize)
  # Create a matrix plot
  cax = ax.matshow(cm, cmap=plt.cm.Blues) # https://matplotlib.org/3.2.0/api/_as_gen/matplotlib.axes.Axes.matshow.html
  fig.colorbar(cax)
  

  # Create classes
  classes = False

  if classes:
    labels = classes
  else:
    labels = np.arange(cm.shape[0])

  # Label the axes
  ax.set(title="Confusion Matrix",
        xlabel="Predicted label",
        ylabel="True label",
        xticks=np.arange(n_classes),
        yticks=np.arange(n_classes),
        xticklabels=labels,
        yticklabels=labels)

  # Set x-axis labels to bottom
  ax.xaxis.set_label_position("bottom")
  ax.xaxis.tick_bottom()

  # Adjust label size
  ax.xaxis.label.set_size(20)
  ax.yaxis.label.set_size(20)
  ax.title.set_size(20)

  # Set threshold for different colors
  threshold = (cm.max() + cm.min()) / 2.

  # Plot the text on each cell
  for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    plt.text(j, i, f"{cm[i, j]} ({cm_norm[i, j]*100:.1f}%)",
            horizontalalignment="center",
            color="white" if cm[i, j] > threshold else "black",
            size=15)

In [ ]:
y_pred_rounded=y_pred.round()

In [ ]:
y_pred_rounded

In [ ]:
plot_confusion_matrix(y_test,y_pred_rounded)

In [ ]:
converter=tf.lite.TFLiteConverter.from_keras_model(model)
converter.target_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
converter.allow_custom_ops=True
converter.experimental_new_converter =True
tflite_model = converter.convert()

In [ ]:
open("./artifacts/heart.tflite","wb").write(tflite_model)